# Одностадийная модель с режимом повторов — полный прогон на GPU

Ноутбук самодостаточный: код проекта записывается в файлы ячейками `%%writefile` ниже.

**Как запустить на Kaggle**

1. *Create → New Notebook → File → Import Notebook* — загрузи этот `.ipynb`.
2. Справа *Settings → Accelerator → GPU T4 x2* (или *P100*). Интернет не нужен.
3. Запуск:
   * быстро проверить — поставь `QUICK = True` в ячейке настроек и *Run All* (пара минут);
   * полный прогон — `QUICK = False`, затем *Save Version → Save & Run All (Commit)*.
     Так ноутбук считается в фоне (до 12 ч), вкладку можно закрыть, результаты останутся во вкладке *Output*.
4. Результаты: `results.zip` в *Output* (внутри `summary.md`, `metrics.csv`, `bootstrap.csv`, `latency*.csv`, логи).

Время на T4/P100: прошлый прогон (base + 1 switch) — ~1 мин обучения на прогон; с абляцией (base + 6 switch) —
оценочно ~3–5 мин на прогон, ≈ 30–45 мин на 9 прогонов.

**Colab:** тоже работает — *Runtime → Change runtime type → GPU*, результаты скачай до отключения сессии.


In [ ]:
import os, sys, time, torch

REQUIRE_GPU = True
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет GPU")
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "Включи GPU: Settings → Accelerator (иначе прогон займёт часы)"

# рабочая папка: на Kaggle — /kaggle/working, иначе — рядом с ноутбуком
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
WORK = os.path.join(BASE, "repeat_switch")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print("работаем в", WORK)
try:
    import tabulate  # noqa: F401  — красивые таблицы в summary.md (без него будет обычный текст)
except ImportError:
    os.system(f"{sys.executable} -m pip install -q tabulate")

import subprocess


def run(args, log):
    """Запускает python-скрипт, печатает вывод по ходу и пишет его в лог (как `| tee`, но кроссплатформенно)."""
    print("$ python", args, flush=True)
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    with open(log, "w", encoding="utf-8") as f, subprocess.Popen(
            f'"{sys.executable}" -u {args}', shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", env=env) as p:
        for line in p.stdout:
            print(line, end="", flush=True)
            f.write(line)
    assert p.returncode == 0, f"процесс упал (код {p.returncode}), см. {log}"


## Код проекта
Сгенерировано `kaggle/build_notebook.py` из исходников.

In [ ]:
%%writefile data.py
"""
Данные и протокол оценки для эксперимента «одностадийная модель + режим повторов».

  make_synthetic()        — синтетический маркетплейс с известной правдой (какие категории расходные),
                            регулятором доли повторов и шумом;
  make_protocol()         — сплит по времени: train / val-окно / test-окно, цели и время запроса;
  load_tecd_marketplace() — загрузчик T-ECD (в экспериментах сейчас не используется).

Формат:
  events: DataFrame[user, item, ts]   ts — время в днях (float), user/item — int с 0
  items:  DataFrame[item, cat, is_consumable]
"""
import glob
import os
from dataclasses import dataclass

import numpy as np
import pandas as pd


def make_synthetic(n_users=4000, n_items=1500, n_cats=40, n_personas=8,
                   consumable_share=0.4, durable_rate=0.5, days=365, brand_switch=0.03,
                   pause_prob=0.3, seasonal_share=0.3, seed=0):
    """
    Регуляторы доли повторов среди целей:
      consumable_share — доля расходных категорий;
      durable_rate     — сколько ещё (сверх одного) товаров длительного пользования в среднем
                         покупается за год в каждой любимой категории (Пуассон).

    Шум, чтобы данные не повторяли устройство модели один в один:
      brand_switch   — на каждой покупке расходника шанс насовсем сменить любимый товар
                       (повтор на уровне категории, но товар новый → explore-цель);
      pause_prob     — доля пользователей с паузой 2–8 недель (отпуск): покупок нет, а после
                       возвращения запас кончился — расходник покупается сразу, «не по графику»;
      seasonal_share — доля категорий длительного пользования с сезонным пиком спроса.
    """
    rng = np.random.default_rng(seed)

    # --- категории: расходные (корм, вода) или длительного пользования (холодильник)
    cat_is_cons = rng.random(n_cats) < consumable_share
    cat_interval = rng.uniform(7, 45, n_cats)          # типичный период повторной покупки, дни
    seasonal = (~cat_is_cons) & (rng.random(n_cats) < seasonal_share)
    cat_peak = rng.uniform(0, days, n_cats)            # день пика для сезонных категорий

    # --- товары: категория + популярность (тяжёлый хвост)
    item_cat = rng.integers(0, n_cats, n_items)
    item_pop = np.minimum(rng.zipf(1.6, n_items), 1000).astype(float)
    items_by_cat = [np.where(item_cat == c)[0] for c in range(n_cats)]

    # --- «персоны»: группы похожих пользователей, чтобы новое было предсказуемо коллаборативно
    persona_w = rng.dirichlet(np.full(n_cats, 0.3), n_personas)

    rows = []
    for u in range(n_users):
        persona = rng.integers(n_personas)
        n_pref = rng.integers(6, 14)
        pref = rng.choice(n_cats, n_pref, replace=False, p=persona_w[persona])

        pause = None
        if rng.random() < pause_prob:
            s = rng.uniform(0, days)
            pause = (s, s + rng.uniform(14, 60))

        def in_pause(t):
            return pause is not None and pause[0] <= t < pause[1]

        for c in pref:
            cand = items_by_cat[c]
            if len(cand) == 0:
                continue
            p = item_pop[cand] / item_pop[cand].sum()

            if cat_is_cons[c]:
                # расходник: любимый товар + личный период покупки
                fav = rng.choice(cand, size=min(2, len(cand)), replace=False, p=p)
                interval = cat_interval[c] * rng.uniform(0.6, 1.6)
                t = rng.uniform(0, interval)
                while t < days:
                    if in_pause(t):
                        t = pause[1] + rng.uniform(0, 3)       # вернулся, запас кончился
                        continue
                    if rng.random() < brand_switch:           # сменил бренд насовсем
                        fav[0] = rng.choice(cand, p=p)
                    item = fav[0]
                    if len(fav) > 1 and rng.random() < 0.2:
                        item = fav[1]
                    if rng.random() < 0.1:                    # иногда пробует новое в категории
                        item = rng.choice(cand, p=p)
                    rows.append((u, int(item), t))
                    t += interval * rng.lognormal(0, 0.25)
            else:
                # длительного пользования: 1 + Poisson(durable_rate) разных товаров за год, без повторов
                bought = set()
                for _ in range(1 + rng.poisson(durable_rate)):
                    item = int(rng.choice(cand, p=p))
                    t = rng.normal(cat_peak[c], 15) % days if seasonal[c] else rng.uniform(0, days)
                    if item in bought or in_pause(t):
                        continue
                    bought.add(item)
                    rows.append((u, item, t))

    events = pd.DataFrame(rows, columns=["user", "item", "ts"])
    events = events.sort_values(["user", "ts"]).reset_index(drop=True)
    items = pd.DataFrame({"item": np.arange(n_items), "cat": item_cat,
                          "is_consumable": cat_is_cons[item_cat]})
    return events, items


# ----------------------------------------------------------------------------- протокол

@dataclass
class Split:
    hist: dict            # user -> (items, times): история до начала окна, по времени
    targets: dict         # user -> np.array различных целевых товаров
    t_query: dict         # user -> время запроса
    events: pd.DataFrame  # все события до начала окна (для статистик: популярность, интервалы)
    t0: float             # начало окна

    @property
    def users(self):
        return np.array(sorted(self.targets))


def user_histories(events):
    """user -> (массив товаров, массив времён), по возрастанию времени."""
    return {u: (g["item"].to_numpy(), g["ts"].to_numpy()) for u, g in events.groupby("user")}


def make_protocol(events, protocol="window", t_val=351.0, t_test=358.0, window=7.0, gap=0.0):
    """
    Протоколы:
      window     — (по умолчанию) запрос в момент t0, цели — все различные товары, купленные
                   в [t0+gap, t0+gap+window). Когда именно человек купит, модели неизвестно.
      next       — старый протокол: одна цель (первая покупка после t0), время запроса = время
                   этой покупки. Утечка: модель знает момент покупки.
      next-start — одна цель, но время запроса = t0. Разница next и next-start = цена утечки.

    Модель учится на событиях < t_val. Val-окно — для ранней остановки и подбора квот/порогов.
    На тесте модель НЕ переобучается, но история пользователя дополняется событиями до t_test.
    """
    assert protocol in ("window", "next", "next-start")
    assert t_val + gap + window <= t_test + 1e-9, "val-окно не должно залезать в тест"

    def build(t0):
        before = events[events.ts < t0]
        hist = user_histories(before)
        w = events[(events.ts >= t0 + gap) & (events.ts < t0 + gap + window)]
        w = w[w.user.isin(list(hist))]                 # нужна хоть какая-то история
        targets, t_query = {}, {}
        if protocol == "window":
            for u, g in w.groupby("user"):
                targets[u] = np.unique(g["item"].to_numpy())
                t_query[u] = t0
        else:
            first = w.sort_values("ts").groupby("user").head(1)
            for u, i, ts in zip(first.user, first.item, first.ts):
                targets[u] = np.array([i])
                t_query[u] = ts if protocol == "next" else t0
        return Split(hist, targets, t_query, before.reset_index(drop=True), t0)

    train = events[events.ts < t_val].reset_index(drop=True)
    return train, build(t_val), build(t_test)


# ----------------------------------------------------------------------------- T-ECD (не используется)

def load_tecd_marketplace(root, user_col="user_id", item_col="item_id", ts_col="timestamp",
                          action_col="action_type", keep_actions=None, max_files=None,
                          top_items=50000, min_events=3):
    """
    Читает marketplace/events/{day}.pq из T-ECD (нужен pyarrow).
    Имена колонок и тип «покупочного» действия проверь в карточке датасета на HF
    (t-tech/T-ECD) и передай аргументами — в статье на Хабре они не указаны.
    keep_actions — список действий, которые считаем «потреблением» (например, переход к покупке).
    top_items    — оставляем самые частые товары: полный softmax по 2.6 млн товаров не влезет.
    """
    files = sorted(glob.glob(os.path.join(root, "marketplace", "events", "*.pq")))
    if max_files:
        files = files[:max_files]
    parts = []
    for f in files:
        df = pd.read_parquet(f)
        if keep_actions is not None:
            df = df[df[action_col].isin(keep_actions)]
        parts.append(df[[user_col, item_col, ts_col]])
    df = pd.concat(parts, ignore_index=True)
    df.columns = ["user_raw", "item_raw", "ts_raw"]

    # время → дни от начала
    if np.issubdtype(df["ts_raw"].dtype, np.datetime64):
        ts = (df["ts_raw"] - df["ts_raw"].min()).dt.total_seconds() / 86400
    else:
        ts = (df["ts_raw"] - df["ts_raw"].min()) / 86400        # если секунды
    df["ts"] = ts.astype(float)

    keep = df["item_raw"].value_counts().index[:top_items]
    df = df[df["item_raw"].isin(keep)]
    n_per_user = df.groupby("user_raw")["item_raw"].transform("size")
    df = df[n_per_user >= min_events].copy()

    df["user"] = df["user_raw"].astype("category").cat.codes
    df["item"] = df["item_raw"].astype("category").cat.codes
    events = df[["user", "item", "ts"]].sort_values(["user", "ts"]).reset_index(drop=True)

    n_items = events["item"].max() + 1
    items = pd.DataFrame({"item": np.arange(n_items), "cat": -1, "is_consumable": np.nan})
    return events, items


In [ ]:
%%writefile metrics.py
"""
Метрики. Главное — считать отдельно по двум типам целей:
  repeat  — пользователь купил то, что уже покупал;
  explore — купил новое.
Общая метрика прячет провал на одной из частей.

У пользователя может быть несколько целей (все покупки в окне), поэтому:
  Recall@k        — среднее по пользователям |hits| / |targets|;
  NDCG@k          — с несколькими релевантными, IDCG по min(|targets|, k);
  HR@k            — доля пользователей, у которых угадана хоть одна цель;
  Recall repeat / explore — микро-усреднение по целям своего типа: Σ hits / Σ targets.
"""
import numpy as np
import pandas as pd


def topk(scores, k):
    """Индексы k лучших по строкам, по убыванию."""
    idx = np.argpartition(-scores, k, axis=1)[:, :k]
    order = np.argsort(-np.take_along_axis(scores, idx, 1), axis=1, kind="stable")
    return np.take_along_axis(idx, order, 1)


def true_repeat_frac(split, users):
    """Доля повторов среди целей каждого пользователя (нужна для оракула)."""
    return np.array([np.isin(split.targets[u], split.hist[u][0]).mean() for u in users])


def _membership(lists, n_items):
    """Ragged-списки товаров → плотная булева матрица (U, N)."""
    M = np.zeros((len(lists), n_items), dtype=bool)
    rows = np.repeat(np.arange(len(lists)), [len(x) for x in lists])
    M[rows, np.concatenate(lists).astype(np.int64)] = True
    return M


def per_user(recs, users, split, items, k=10):
    """recs: (U, >=k) — рекомендации для users (в том же порядке). Возвращает метрики по пользователям."""
    n_items = len(items)
    is_cons = items["is_consumable"].to_numpy().astype(float)
    has_truth = not np.isnan(is_cons).any()
    disc = 1.0 / np.log2(np.arange(2, k + 2))

    T = _membership([split.targets[u] for u in users], n_items)   # цели
    H = _membership([split.hist[u][0] for u in users], n_items)   # история
    top = np.asarray(recs)[:, :k]
    r = np.arange(len(users))[:, None]
    hit = T[r, top]                                  # (U, k): позиция выдачи попала в цель
    in_hist = H[r, top]
    n_t = T.sum(1)
    df = pd.DataFrame({
        "user": users, "n_t": n_t, "n_rep_t": (T & H).sum(1),
        "hits": hit.sum(1), "hits_rep": (hit & in_hist).sum(1), "hits_exp": (hit & ~in_hist).sum(1),
        "ndcg": (hit * disc).sum(1) / np.cumsum(disc)[np.minimum(n_t, k) - 1],
        "n_rep_in_top": in_hist.sum(1),
        "wasted": (in_hist & (is_cons[top] == 0)).sum(1) if has_truth else np.nan,
    })
    df["recall"] = df.hits / df.n_t
    df["hit"] = (df.hits > 0).astype(float)
    df["repeat_frac"] = df.n_rep_t / df.n_t
    return df


def auc(score, label):
    """ROC AUC через ранги (Манн–Уитни)."""
    label = np.asarray(label, bool)
    n1, n0 = label.sum(), (~label).sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    r = pd.Series(score).rank().to_numpy()
    return (r[label].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def summarize(df, p_switch=None, k=10):
    n_exp_t = (df.n_t - df.n_rep_t).sum()
    out = {
        f"Recall@{k}": df.recall.mean(),
        f"NDCG@{k}": df.ndcg.mean(),
        f"HR@{k}": df.hit.mean(),
        "Recall repeat": df.hits_rep.sum() / max(df.n_rep_t.sum(), 1),
        "Recall explore": df.hits_exp.sum() / max(n_exp_t, 1),
        "repeat share (true)": df.n_rep_t.sum() / df.n_t.sum(),
        "repeat slots": df.n_rep_in_top.mean() / k,
        "wasted slots": df.wasted.mean(),
    }
    if p_switch is not None:
        # p_switch — прогноз «следующая покупка будет повтором»; правда — доля повторов в целях
        p = np.asarray(p_switch)
        y = (df.repeat_frac >= 0.5).to_numpy()
        out["switch AUC"] = auc(p, y)
        out["switch Brier"] = float(np.mean((p - df.repeat_frac.to_numpy()) ** 2))
        out["p | repeat"] = p[y].mean() if y.any() else np.nan
        out["p | explore"] = p[~y].mean() if (~y).any() else np.nan
    return out


def bootstrap_diff(a, b, col="recall", n_boot=2000, seed=0):
    """Парный бутстрап по пользователям: среднее(a − b) и 95% доверительный интервал."""
    d = a[col].to_numpy() - b[col].to_numpy()
    rng = np.random.default_rng(seed)
    means = d[rng.integers(0, len(d), (n_boot, len(d)))].mean(1)
    return float(d.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


In [ ]:
%%writefile mixing.py
"""
Правила смешивания двух голов — общие для эвристик и нейросети, чтобы сравнение было честным.

Components:
  rep (U, N) — распределение repeat-головы по товарам истории (0 вне истории);
  exp (U, N) — распределение explore-головы по товарам НЕ из истории (0 на истории);
  p   (U,)   — вероятность режима повторов на этот запрос.

Правила:
  switch(p)  — мягкая смесь p·rep + (1−p)·exp (выученный переключатель);
  quota(q)   — q лучших из repeat-головы + остальное из explore (q общий или свой на пользователя);
  route(thr) — жёсткий роутинг: p > thr → вся выдача из repeat-головы, иначе — вся из explore
               (исходная идея «классификатор запроса → отдельная модель»).
"""
from dataclasses import dataclass

import numpy as np

from metrics import per_user, topk


@dataclass
class Components:
    rep: np.ndarray
    exp: np.ndarray
    p: np.ndarray


def switch(c, k, p=None):
    p = (c.p if p is None else np.asarray(p))[:, None]
    return topk(p * c.rep + (1 - p) * c.exp, k)


def quota(c, k, q):
    q = np.broadcast_to(np.asarray(q, dtype=float), (len(c.p),))
    rep_top, exp_top = topk(c.rep, k), topk(c.exp, k)
    n_pos = (c.rep > 0).sum(1)                 # меньше k повторов → добиваем explore
    out = np.empty((len(c.p), k), dtype=np.int64)
    for u in range(len(c.p)):
        m = int(min(q[u], n_pos[u], k))
        out[u, :m] = rep_top[u, :m]
        out[u, m:] = exp_top[u, :k - m]
    return out


def route(c, k, thr):
    return quota(c, k, np.where(c.p > thr, k, 0))


def tune(make_recs, grid, split, users, items, k):
    """Лучшее значение параметра по Recall@k на val. Возвращает (значение, recall)."""
    scores = [(g, per_user(make_recs(g), users, split, items, k).recall.mean()) for g in grid]
    return max(scores, key=lambda s: s[1])


def tune_quota(c, split, users, items, k):
    return tune(lambda q: quota(c, k, q), range(k + 1), split, users, items, k)[0]


def tune_route(c, split, users, items, k):
    grid = [-1.0, 2.0] + list(np.unique(np.quantile(c.p, np.linspace(0.05, 0.95, 19))))
    return tune(lambda t: route(c, k, t), grid, split, users, items, k)[0]


In [ ]:
%%writefile baselines.py
"""
Сильные простые бейзлайны. В задачах с повторными покупками они часто обгоняют нейросети
(см. Li et al., «A Next Basket Recommendation Reality Check»), поэтому без них вывод
«нейросеть лучше» ничего не стоит.

  pop       — популярность за последние 30 дней (нижняя граница);
  freq+pop  — товары из истории по частоте (при равенстве — по свежести), остаток — популярное;
  tifu-knn  — упрощённый TIFU-KNN: вектор пользователя = покупки с затуханием по времени,
              прогноз = α·свой вектор + (1−α)·средний вектор K ближайших соседей (косинус).
"""
import numpy as np

from metrics import topk


def history_matrix(split, users, n_items, tau=None, t_ref=None):
    """(U, N): число покупок (tau=None) или Σ exp(−(t_ref − t)/tau)."""
    M = np.zeros((len(users), n_items))
    for r, u in enumerate(users):
        it, ts = split.hist[u]
        w = np.ones(len(it)) if tau is None else np.exp(-((split.t0 if t_ref is None else t_ref) - ts) / tau)
        np.add.at(M[r], it, w)
    return M


def recent_pop_scores(split, n_items, days=30):
    ev = split.events[split.events.ts >= split.t0 - days]
    return np.bincount(ev["item"], minlength=n_items).astype(float)


def pop(split, users, n_items, k):
    s = recent_pop_scores(split, n_items)
    return np.tile(topk(s[None], k), (len(users), 1))


def freq_pop(split, users, n_items, k):
    cnt = history_matrix(split, users, n_items)
    fresh = history_matrix(split, users, n_items, tau=30.0)        # ≤ cnt → только тай-брейк
    p = recent_pop_scores(split, n_items)
    p = 0.999 * p / (p.max() + 1e-12)                              # в [0, 1): ниже любого повтора
    scores = np.where(cnt > 0, 1.0 + cnt + 1e-3 * fresh, p[None, :])
    return topk(scores, k)


def tifu_knn(split, users, n_items, k, tau=30.0, n_neighbors=100, alpha=0.7):
    pool = np.array(sorted(split.hist))                            # соседи — все, у кого есть история
    V = history_matrix(split, pool, n_items, tau=tau)
    V /= np.linalg.norm(V, axis=1, keepdims=True) + 1e-12
    row = {u: r for r, u in enumerate(pool)}
    Vu = V[[row[u] for u in users]]
    S = Vu @ V.T
    S[np.arange(len(users)), [row[u] for u in users]] = -np.inf     # себя не считаем соседом
    nb = np.argpartition(-S, n_neighbors, axis=1)[:, :n_neighbors]
    W = np.zeros_like(S)
    W[np.arange(len(users))[:, None], nb] = 1.0 / n_neighbors
    neigh = W @ V                                                   # среднее по соседям
    return topk(alpha * Vu + (1 - alpha) * neigh, k)


TIFU_GRID = [(tau, kn, a) for tau in (15.0, 60.0) for kn in (50, 300) for a in (0.5, 0.8)]


In [ ]:
%%writefile heuristics.py
"""
Эвристические головы (без нейросетей, только numpy). Отдают те же Components, что и нейросеть,
поэтому все правила смешивания (quota / switch / route / oracle) применяются одинаково.

  base    — одна шкала «любимые категории × популярность», купленное не исключаем;
  repeat  — насколько «пора» купить товар из истории снова (по медианному интервалу повтора);
  explore — та же шкала категорий × популярность, но без товаров из истории;
  p       — «созрел» ли самый готовый к повтору товар.
"""
import numpy as np

from mixing import Components


def item_intervals(events):
    """Медианный интервал между повторными покупками одного товара (по всем пользователям)."""
    tr = events.sort_values(["user", "item", "ts"])
    diffs = tr.groupby(["user", "item"])["ts"].diff().dropna()
    diffs = diffs[diffs > 0]
    med = tr.loc[diffs.index].assign(d=diffs).groupby("item")["d"].median()
    return med.to_dict()


def repeat_scores(h_items, h_times, t_now, intervals):
    """
    Для каждого товара из истории: насколько «пора» купить снова, в [0, 1].
    1 — с последней покупки прошло ровно типичное время; рано или сильно поздно — меньше.
    Товары, которые никто не покупал повторно (нет интервала), получают 0.
    """
    last = {}
    for i, t in zip(h_items, h_times):
        last[i] = t
    scores = {}
    for i, t_last in last.items():
        if i not in intervals:
            continue
        age = max(t_now - t_last, 1e-3)
        scores[i] = float(np.exp(-abs(np.log(age / intervals[i]))))
    return scores


def explore_scores(h_items, item_cat, item_pop, n_cats):
    """Аффинность к категориям из истории × популярность товара (сглаженно)."""
    cat_aff = np.ones(n_cats)                 # сглаживание
    np.add.at(cat_aff, item_cat[h_items], 5)
    cat_aff /= cat_aff.sum()
    return cat_aff[item_cat] * item_pop


def heur_components(split, users, items):
    """Возвращает (base_scores (U, N), Components). Статистики — только по событиям до окна."""
    item_cat = items["cat"].to_numpy()
    if (item_cat < 0).all():                  # нет категорий — одна общая
        item_cat = np.zeros_like(item_cat)
    n_cats, n_items = item_cat.max() + 1, len(items)
    item_pop = np.bincount(split.events["item"], minlength=n_items).astype(float) + 1
    item_pop /= item_pop.sum()
    intervals = item_intervals(split.events)

    U = len(users)
    base = np.zeros((U, n_items))
    rep = np.zeros((U, n_items))
    exp = np.zeros((U, n_items))
    p = np.zeros(U)
    for r, u in enumerate(users):
        h_items, h_times = split.hist[u]
        e = explore_scores(h_items, item_cat, item_pop, n_cats)
        base[r] = e
        e = e.copy()
        e[np.unique(h_items)] = 0
        exp[r] = e / e.sum()
        sc = repeat_scores(h_items, h_times, split.t_query[u], intervals)
        if sc:
            idx = np.fromiter(sc.keys(), dtype=np.int64)
            val = np.fromiter(sc.values(), dtype=float)
            rep[r, idx] = val / val.sum()
            p[r] = val.max()
    return base, Components(rep, exp, p)


In [ ]:
%%writefile model.py
"""
Одностадийная генеративная модель: трансформер над историей покупок → распределение над всем каталогом.

Режимы:
  base   — одна голова: softmax над всеми товарами (как обычная одностадийная модель);
  switch — две головы + переключатель (идея RepeatNet):
             repeat-голова  — указатель (pointer) на товары из истории, учитывает «возраст» покупки;
             explore-голова — softmax только по товарам, которых нет в истории;
             p = sigmoid(w·h) — вероятность режима повторов на этот запрос;
             P(item) = p · P_repeat(item) + (1 − p) · P_explore(item).
  fixed30 использует ту же обученную switch-модель, но смешивает головы жёсткой квотой при выдаче.

Борьба с «потраченными слотами» (повтор уже купленного холодильника), опции repeat-головы:
  rep_bias  — обучаемый скаляр «повторяемости» на каждый товар, добавляется к логиту указателя;
  rep_prior — логит сглаженной доли повторных покупок товара из train (счётчик, не обучается)
              с обучаемым весом; обобщается на редкие товары лучше, чем rep_bias;
  rep_null  — «пустой» слот указателя: если в истории нечего повторять, масса уходит в него
              и перетекает в explore. Указатель обязан распределить 1.0 по истории, и без null-слота
              даже нерелевантные товары истории получают заметную массу — а на выдаче
              p·P_repeat легко обгоняет (1−p)·P_explore, размазанную по всему каталогу.
              С null-слотом: P = p·P_rep + (1 − p + p·p_null)·P_explore, эффективный переключатель p·(1 − p_null).
Без опций модель совпадает с исходной.

Индексы товаров в модели: 1..n_items, 0 — паддинг. Паддинг справа (последовательность начинается с 0-й позиции).
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

N_BUCKETS = 32
NEG = -1e9


def time_bucket(dt_days):
    """Лог-корзины по времени: ~час, несколько часов, день, неделя, месяц... (до ~года)."""
    hours = torch.clamp(dt_days, min=0) * 24.0
    b = torch.floor(torch.log2(hours + 1.0) * 2).long()
    return torch.clamp(b, 0, N_BUCKETS - 1)


class OneStageRec(nn.Module):
    def __init__(self, n_items, mode="switch", d=64, n_layers=2, n_heads=2, max_len=50, dropout=0.1,
                 rep_bias=False, rep_null=False, rep_prior=None):
        super().__init__()
        assert mode in ("base", "switch")
        self.mode = mode
        self.n_items = n_items
        self.d = d
        self.item_emb = nn.Embedding(n_items + 1, d, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d)
        self.gap_emb = nn.Embedding(N_BUCKETS, d)        # пауза между соседними покупками
        self.query_emb = nn.Embedding(N_BUCKETS, d)      # сколько прошло от последней покупки до запроса
        layer = nn.TransformerEncoderLayer(d, n_heads, 4 * d, dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.dropout = nn.Dropout(dropout)
        if mode == "switch":
            self.age_emb = nn.Embedding(N_BUCKETS, d)    # «возраст» покупки из истории на момент запроса
            self.rep_query = nn.Linear(d, d)
            self.switch = nn.Linear(d, 1)
            self.rep_bias = nn.Embedding(n_items + 1, 1) if rep_bias else None
            self.null_key = nn.Parameter(torch.zeros(d)) if rep_null else None
            if rep_prior is not None:          # (n_items + 1,), индекс 0 — паддинг
                self.register_buffer("rep_prior", torch.as_tensor(rep_prior, dtype=torch.float32))
                self.prior_w = nn.Parameter(torch.tensor(1.0))
            else:
                self.rep_prior = None
        # Эмбеддинги N(0,1) при d=64 дают логиты со std ~10 на старте (softmax насыщен).
        # Стандартная инициализация SASRec/BERT: std=0.02.
        for emb in self.modules():
            if isinstance(emb, nn.Embedding):
                nn.init.normal_(emb.weight, std=0.02)
                if emb.padding_idx is not None:
                    with torch.no_grad():
                        emb.weight[emb.padding_idx].zero_()
        if mode == "switch" and self.rep_bias is not None:
            nn.init.zeros_(self.rep_bias.weight)     # старт = исходная модель

    def encode(self, items, times, query_times):
        """
        items, times:  (B, L) — история; query_times: (B, L) — момент запроса для каждой позиции
        (в обучении это время следующей покупки, в инференсе — «сейчас»).
        Возвращает q: (B, L, d) — состояние пользователя к моменту запроса.
        """
        B, L = items.shape
        prev = torch.cat([times[:, :1], times[:, :-1]], dim=1)
        pos = torch.arange(L, device=items.device).unsqueeze(0).expand(B, L)
        x = self.item_emb(items) + self.pos_emb(pos) + self.gap_emb(time_bucket(times - prev))
        x = self.dropout(x)
        causal = torch.triu(torch.ones(L, L, dtype=torch.bool, device=items.device), diagonal=1)
        h = self.norm(self.encoder(x, mask=causal))
        return h + self.query_emb(time_bucket(query_times - times))

    def forward(self, items, times, query_times, pre_seen=None):
        """
        base:   возвращает (log_probs, None, None, None)
        switch: возвращает (log_probs смеси, p_repeat (нормированная), p_explore, p_switch (эффективный))
        log_probs/p_*: (B, L, n_items + 1), p_switch: (B, L, 1)
        pre_seen: (B, n_items + 1) bool — товары, купленные ДО окна последних max_len покупок.
                  Explore-голова по определению — «чего не было в истории», поэтому исключает и их;
                  без этого холодильник, выпавший из окна, рекомендуется заново.
        """
        q = self.encode(items, times, query_times)
        logits = q @ self.item_emb.weight.T                       # (B, L, N+1)
        pad_col = torch.zeros_like(logits, dtype=torch.bool)
        pad_col[..., 0] = True
        logits = logits.masked_fill(pad_col, NEG)

        if self.mode == "base":
            return F.log_softmax(logits, dim=-1), None, None, None

        B, L = items.shape
        # --- repeat-голова: внимание позиции i на покупки j <= i с учётом их возраста к моменту запроса
        age = query_times.unsqueeze(2) - times.unsqueeze(1)       # (B, L_i, L_j)
        # att_ij = rq_i · (e_j + age_emb[bucket_ij]) = rq_i·e_j + (rq_i·age_emb)[bucket_ij]
        # — без тензора (B, L, L, d): вторая часть берётся gather'ом из (B, L, N_BUCKETS)
        rq = self.rep_query(q)                                                     # (B, L, d)
        att_item = rq @ self.item_emb(items).transpose(1, 2)                      # (B, L, L)
        att_age = (rq @ self.age_emb.weight.T).gather(2, time_bucket(age))        # (B, L, L)
        att = (att_item + att_age) / math.sqrt(self.d)
        if self.rep_bias is not None:                              # «повторяемость» товара j
            att = att + self.rep_bias(items).squeeze(-1).unsqueeze(1)
        if self.rep_prior is not None:
            att = att + self.prior_w * self.rep_prior[items].unsqueeze(1)
        lower = torch.tril(torch.ones(L, L, dtype=torch.bool, device=items.device))
        valid = lower.unsqueeze(0) & (items != 0).unsqueeze(1)    # (B, L_i, L_j)
        att = att.masked_fill(~valid, NEG)
        if self.null_key is not None:
            null = (rq @ self.null_key / math.sqrt(self.d)).unsqueeze(-1)            # (B, L, 1)
            w = torch.softmax(torch.cat([att, null], dim=-1), dim=-1)
            w, p_null = w[..., :L], w[..., L:]
        else:
            w = torch.softmax(att, dim=-1)
            p_null = torch.zeros_like(att[..., :1])

        # складываем веса позиций в вероятности товаров (один товар мог встречаться несколько раз)
        index = items.unsqueeze(1).expand(B, L, L).contiguous()
        p_rep = torch.zeros_like(logits).scatter_add(2, index, w)
        seen = torch.zeros_like(logits).scatter_add(2, index, valid.float()) > 0
        if pre_seen is not None:
            seen = seen | pre_seen.unsqueeze(1)

        # --- explore-голова: только товары, которых нет в истории
        p_exp = torch.softmax(logits.masked_fill(seen, NEG), dim=-1)

        # --- переключатель
        p_sw = torch.sigmoid(self.switch(q))                     # (B, L, 1)
        probs = p_sw * p_rep + (1 - p_sw + p_sw * p_null) * p_exp
        # для правил смешивания: нормированная repeat-голова и эффективный переключатель,
        # p_eff·rep_norm + (1 − p_eff)·exp == probs
        p_eff = p_sw * (1 - p_null)
        rep_norm = p_rep / (1 - p_null).clamp_min(1e-9)
        return torch.log(probs + 1e-10), rep_norm, p_exp, p_eff


In [ ]:
%%writefile train.py
"""
Эксперимент: одностадийная модель без режимов vs квота повторов vs выученный переключатель
vs жёсткий роутинг, против сильных простых бейзлайнов.

Протокол (см. data.make_protocol):
  * модель учится на событиях < t_val, ранняя остановка и подбор квоты/порога/гиперпараметров
    бейзлайнов — на val-окне; на тесте ничего не подбирается;
  * запрос — в начале окна, цели — все покупки в окне (время покупки модели неизвестно);
  * несколько сидов (сид меняет и данные, и инициализацию), парный бутстрап по пользователям.

Запуск:
  python train.py                                         # 3 режима доли повторов × 3 сида
  python train.py --regimes high --seeds 0                # один режим, быстро
  python train.py --protocol next --seeds 0 --out results/leak_next   # старый протокол с утечкой
  python train.py --skip-nn                               # быстро, только эвристики и бейзлайны
"""
import argparse
import copy
import json
import os
import random
import time

import numpy as np
import pandas as pd
import torch

import baselines as bl
from data import make_protocol, make_synthetic
from heuristics import heur_components
from metrics import bootstrap_diff, per_user, summarize, topk, true_repeat_frac
from mixing import Components, quota, route, switch, tune, tune_quota, tune_route
from model import OneStageRec

# режимы синтетики: (consumable_share, durable_rate) → доля повторов среди целей теста
REGIMES = {"low": (0.10, 15.0),     # 0.13–0.37 (по сидам 0–2)
           "mid": (0.15, 15.0),     # 0.23–0.54
           "high": (0.40, 0.5)}     # 0.82–0.88 (исходная синтетика)

# --------------------------------------------------------------------------- нейросеть


def build_train_sequences(train, max_len):
    """
    Окна по max_len+1 покупок с конца истории; каждый переход попадает в обучение ровно один раз.
    Третий элемент — товары, купленные до окна (для маски explore-головы, см. model.forward).
    """
    seqs = []
    for _, g in train.groupby("user"):
        it = g["item"].to_numpy() + 1             # +1: 0 зарезервирован под паддинг
        ts = g["ts"].to_numpy()
        end = len(it)
        while end >= 2:
            start = max(0, end - (max_len + 1))
            seqs.append((it[start:end], ts[start:end], np.unique(it[:start])))
            if start == 0:
                break
            end = start + 1
    return seqs


def pad_batch(batch, length, n_items=None):
    """
    Паддинг справа. Время в паддинге = последнее реальное время (чтобы не было мусорных интервалов).
    Если задан n_items — ещё и маска (B, n_items + 1) товаров, купленных до окна.
    """
    items = np.zeros((len(batch), length), dtype=np.int64)
    times = np.zeros((len(batch), length), dtype=np.float32)
    lens = np.zeros(len(batch), dtype=np.int64)
    pre = np.zeros((len(batch), n_items + 1), dtype=bool) if n_items is not None else None
    for b, (it, ts, before) in enumerate(batch):
        n = len(it)
        items[b, :n] = it
        times[b, :n] = ts
        times[b, n:] = ts[-1]
        lens[b] = n
        if pre is not None:
            pre[b, before] = True
    pre = torch.from_numpy(pre) if pre is not None else None
    return torch.from_numpy(items), torch.from_numpy(times), torch.from_numpy(lens), pre


@torch.no_grad()
def nn_outputs(model, split, users, args):
    """Прогноз на момент запроса. base → scores (U, N); switch → Components."""
    model.eval()
    outs = []
    for s in range(0, len(users), args.batch):
        bu = users[s:s + args.batch]
        batch = [(split.hist[u][0][-args.max_len:] + 1, split.hist[u][1][-args.max_len:],
                  np.unique(split.hist[u][0][:-args.max_len] + 1)) for u in bu]
        n_mask = model.n_items if getattr(model, "full_mask", False) else None
        items, times, lens, pre = pad_batch(batch, args.max_len, n_mask)
        t_now = torch.tensor([split.t_query[u] for u in bu], dtype=torch.float32)
        t_now = t_now.unsqueeze(1).expand_as(times).contiguous()
        items, times, t_now = items.to(args.device), times.to(args.device), t_now.to(args.device)
        pre = pre.to(args.device) if pre is not None else None
        logp, p_rep, p_exp, p_sw = model(items, times, t_now, pre)
        rows, last = torch.arange(len(bu), device=args.device), (lens - 1).to(args.device)
        if model.mode == "base":
            outs.append(logp[rows, last, 1:].cpu().numpy())
        else:
            outs.append((p_rep[rows, last, 1:].cpu().numpy(), p_exp[rows, last, 1:].cpu().numpy(),
                         p_sw[rows, last, 0].cpu().numpy()))
    if model.mode == "base":
        return np.concatenate(outs)
    return Components(*(np.concatenate([o[i] for o in outs]) for i in range(3)))


def nn_recs(model, split, users, args):
    out = nn_outputs(model, split, users, args)
    return topk(out, args.k) if model.mode == "base" else switch(out, args.k)


def repeat_prior(train, n_items, alpha=5.0):
    """
    Логит сглаженной доли повторных покупок товара по train: (повторы + α·g) / (покупки + α),
    повторы = покупки − число различных покупателей, g — общая доля. Индекс 0 — паддинг.
    """
    n = np.bincount(train["item"], minlength=n_items).astype(float)
    buyers = train.groupby("item")["user"].nunique().reindex(range(n_items), fill_value=0).to_numpy()
    rep = n - buyers
    g = rep.sum() / max(n.sum(), 1.0)
    r = np.clip((rep + alpha * g) / (n + alpha), 1e-4, 1 - 1e-4)
    return np.concatenate([[0.0], np.log(r / (1 - r))])


# варианты switch-модели для абляции: имя → опции (см. model.py)
#   mask  — explore-голова исключает ВСЮ историю, а не только окно последних max_len покупок
#           (главная причина wasted slots: 98% из них — товары, выпавшие из окна);
#   bias / null / prior — опции repeat-головы.
_ALL = {"rep_bias": True, "rep_null": True, "rep_prior": True}
SWITCH_VARIANTS = {"switch": {}, "mask": {"full_mask": True},
                   "mask+bias": {"full_mask": True, "rep_bias": True},
                   "mask+null": {"full_mask": True, "rep_null": True},
                   "mask+prior": {"full_mask": True, "rep_prior": True},
                   "mask+all": {"full_mask": True, **_ALL}}


def variant_prefix(sv):
    return "nn" if sv == "switch" else f"nn+{sv}"


def model_kwargs(sv, prior):
    kw = dict(SWITCH_VARIANTS[sv])
    if kw.pop("rep_prior", False):
        kw["rep_prior"] = prior
    return kw


def train_model(seqs, n_items, mode, args, val, val_users, items, full_mask=False, **model_kw):
    """Ранняя остановка по Recall@k на val-окне; возвращается лучшая эпоха."""
    model = OneStageRec(n_items, mode=mode, d=args.dim, n_layers=args.layers,
                        max_len=args.max_len, **model_kw).to(args.device)
    model.full_mask = full_mask
    lr = args.lr if args.lr is not None else getattr(args, f"lr_{mode}")
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best, best_state, best_epoch, bad = -1.0, None, 0, 0
    for epoch in range(args.max_epochs):
        model.train()
        random.shuffle(seqs)
        total, count = 0.0, 0
        for s in range(0, len(seqs), args.batch):
            items_b, times, _, pre = pad_batch(seqs[s:s + args.batch], args.max_len + 1,
                                               n_items if full_mask else None)
            items_b, times = items_b.to(args.device), times.to(args.device)
            pre = pre.to(args.device) if pre is not None else None
            inp, tgt = items_b[:, :-1], items_b[:, 1:]
            logp, _, _, _ = model(inp, times[:, :-1], times[:, 1:], pre)
            lp = logp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
            mask = (tgt != 0).float()
            loss = -(lp * mask).sum() / mask.sum()
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * mask.sum().item()
            count += mask.sum().item()
        rec = per_user(nn_recs(model, val, val_users, args), val_users, val, items, args.k).recall.mean()
        print(f"    [{mode}] epoch {epoch + 1}  train NLL {total / count:.4f}  val Recall@{args.k} {rec:.4f}")
        if rec > best + 1e-4:
            best, best_state, best_epoch, bad = rec, copy.deepcopy(model.state_dict()), epoch + 1, 0
        else:
            bad += 1
            if bad >= args.patience:
                break
    if best_epoch == args.max_epochs:
        print(f"    [{mode}] ВНИМАНИЕ: лучшая эпоха = лимит {args.max_epochs}, модель могла не сойтись")
    model.load_state_dict(best_state)
    return model, best_epoch, best


# --------------------------------------------------------------------------- варианты


def two_head_variants(prefix, base_scores, comps, split, users, k, q, thr, full=True):
    """Все правила смешивания поверх одних и тех же голов (full=False — только главные, для абляции)."""
    frac = true_repeat_frac(split, users)
    v = {}
    if base_scores is not None:
        v[f"{prefix}/base"] = (topk(base_scores, k), None)
    if full:
        v[f"{prefix}/fixed30"] = (quota(comps, k, round(0.3 * k)), None)
    v[f"{prefix}/fixed-tuned"] = (quota(comps, k, q), None)
    v[f"{prefix}/switch"] = (switch(comps, k), comps.p)
    if full:
        v[f"{prefix}/route"] = (route(comps, k, thr), comps.p)
    v[f"{prefix}/oracle-switch"] = (switch(comps, k, p=frac), None)    # знает долю повторов в целях
    if full:
        v[f"{prefix}/oracle-quota"] = (quota(comps, k, np.round(k * frac)), None)
    return v


# --------------------------------------------------------------------------- латентность


def _sync(args):
    if str(args.device).startswith("cuda"):
        torch.cuda.synchronize()


def timed(fn, args):
    """(результат, секунды) — с синхронизацией GPU."""
    _sync(args)
    t = time.perf_counter()
    out = fn()
    _sync(args)
    return out, time.perf_counter() - t


def online_latency(model, split, users, args, n=200, warmup=20, seed=0):
    """
    Онлайн-сценарий: один запрос = один пользователь (batch=1), полный путь от истории
    до топ-k: паддинг → прямой проход → смешивание голов → top-k. Возвращает p50/p95/p99, мс.
    """
    one = argparse.Namespace(**{**vars(args), "batch": 1})
    sample = np.random.default_rng(seed).choice(users, n + warmup)
    ms = []
    for i, u in enumerate(sample):
        _, dt = timed(lambda: nn_recs(model, split, np.array([u]), one), args)
        if i >= warmup:
            ms.append(dt * 1000)
    return {f"online p{q} ms": float(np.percentile(ms, q)) for q in (50, 95, 99)}


def run_one(regime, seed, args):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    k = args.k
    cons, dur = REGIMES[regime]
    events, items = make_synthetic(n_users=args.n_users, consumable_share=cons, durable_rate=dur,
                                   seed=seed)
    n_items = len(items)
    train, val, test = make_protocol(events, args.protocol, args.t_val, args.t_test, args.window, args.gap)
    vu, tu = val.users, test.users
    print(f"\n=== regime={regime} seed={seed}: events {len(events)}, train {len(train)}, "
          f"val users {len(vu)}, test users {len(tu)}")

    chosen = {}
    V = {"val": {}, "test": {}}      # split -> name -> (recs, p_switch)
    lat = []                          # латентность и стоимость
    base_row = {"regime": regime, "seed": seed, "device": str(args.device), "users": len(tu)}

    def lat_batch(name, sec, **extra):
        lat.append({**base_row, "variant": name, "batch ms/user": 1000 * sec / len(tu), **extra})

    # --- простые бейзлайны (время — полный расчёт выдачи для всех тестовых пользователей)
    g, _ = tune(lambda g: bl.tifu_knn(val, vu, n_items, k, *g), bl.TIFU_GRID, val, vu, items, k)
    chosen["tifu (tau, K, alpha)"] = g
    for name, sp, us in (("val", val, vu), ("test", test, tu)):
        for var, fn in (("pop", lambda: bl.pop(sp, us, n_items, k)),
                        ("freq+pop", lambda: bl.freq_pop(sp, us, n_items, k)),
                        ("tifu-knn", lambda: bl.tifu_knn(sp, us, n_items, k, *g))):
            recs, sec = timed(fn, args)
            V[name][var] = (recs, None)
            if name == "test":
                lat_batch(var, sec)

    # --- эвристические головы
    hb_v, hc_v = heur_components(val, vu, items)
    (hb_t, hc_t), sec = timed(lambda: heur_components(test, tu, items), args)
    lat_batch("heur/switch", sec + timed(lambda: switch(hc_t, k), args)[1])
    q, thr = tune_quota(hc_v, val, vu, items, k), tune_route(hc_v, val, vu, items, k)
    chosen["heur q"], chosen["heur thr"] = q, float(thr)
    V["val"].update(two_head_variants("heur", hb_v, hc_v, val, vu, k, q, thr))
    V["test"].update(two_head_variants("heur", hb_t, hc_t, test, tu, k, q, thr))

    # --- нейросеть: base + варианты switch-модели (абляция repeat-головы)
    if not args.skip_nn:
        seqs = build_train_sequences(train, args.max_len)
        prior = repeat_prior(train, n_items)
        models = {}
        t0 = time.time()
        models["base"], chosen["nn/base epoch"], _ = train_model(list(seqs), n_items, "base", args, val, vu, items)
        for sv in args.switch_variants:
            models[sv], chosen[f"{variant_prefix(sv)} epoch"], _ = train_model(
                list(seqs), n_items, "switch", args, val, vu, items, **model_kwargs(sv, prior))
        chosen["nn train sec"] = round(time.time() - t0)

        nb_v = nn_outputs(models["base"], val, vu, args)
        nb_t, sec = timed(lambda: nn_outputs(models["base"], test, tu, args), args)
        lat_batch("nn/base", sec + timed(lambda: topk(nb_t, k), args)[1],
                  params=sum(p.numel() for p in models["base"].parameters()),
                  **online_latency(models["base"], test, tu, args, seed=seed))
        for sv in args.switch_variants:
            m, pre = models[sv], variant_prefix(sv)
            nc_v = nn_outputs(m, val, vu, args)
            nc_t, sec = timed(lambda: nn_outputs(m, test, tu, args), args)
            lat_batch(f"{pre}/switch", sec + timed(lambda: switch(nc_t, k), args)[1],
                      params=sum(p.numel() for p in m.parameters()),
                      **online_latency(m, test, tu, args, seed=seed))
            q, thr = tune_quota(nc_v, val, vu, items, k), tune_route(nc_v, val, vu, items, k)
            chosen[f"{pre} q"], chosen[f"{pre} thr"] = q, float(thr)
            full = sv == "switch"
            V["val"].update(two_head_variants(pre, nb_v if full else None, nc_v, val, vu, k, q, thr, full))
            V["test"].update(two_head_variants(pre, nb_t if full else None, nc_t, test, tu, k, q, thr, full))

    # --- метрики
    rows, per = [], {"val": {}, "test": {}}
    for name, sp, us in (("val", val, vu), ("test", test, tu)):
        for var, (recs, p) in V[name].items():
            df = per_user(recs, us, sp, items, k)
            per[name][var] = df
            rows.append({"regime": regime, "seed": seed, "split": name, "variant": var,
                         **summarize(df, p, k)})

    # --- значимость: эталон против конкурентов (лучший не-нейросетевой выбран по val, не по test)
    ref = "nn/switch" if not args.skip_nn else "heur/switch"
    non_nn = [v for v in per["val"] if not v.startswith("nn") and "oracle" not in v]
    best_simple = max(non_nn, key=lambda v: per["val"][v].recall.mean())
    rivals = [best_simple] + [v for v in ("nn/base", "nn/fixed-tuned", "nn/route", "heur/switch")
                              if v in per["test"] and v != ref and v != best_simple]
    pairs = [(ref, rv, col) for rv in rivals for col in ("recall", "ndcg")]
    # абляция: каждый вариант repeat-головы против исходного switch, в т.ч. по потраченным слотам
    pairs += [(f"{variant_prefix(sv)}/switch", ref, col) for sv in getattr(args, "switch_variants", [])
              if sv != "switch" and not args.skip_nn for col in ("recall", "ndcg", "wasted")]
    boot = []
    for a, b, col in pairs:
        d, lo, hi = bootstrap_diff(per["test"][a], per["test"][b], col, seed=seed)
        boot.append({"regime": regime, "seed": seed, "ref": a, "rival": b,
                     "rival is best simple (val)": b == best_simple,
                     "metric": col, "diff": d, "ci_lo": lo, "ci_hi": hi})
    print("    подобрано на val:", chosen)
    return rows, boot, {"regime": regime, "seed": seed, **{k_: str(v) for k_, v in chosen.items()}}, lat


# --------------------------------------------------------------------------- отчёт

SHOW = ["Recall@10", "NDCG@10", "Recall repeat", "Recall explore", "repeat slots", "wasted slots",
        "switch AUC", "switch Brier"]


def to_md(df, **kw):
    """Markdown-таблица; без пакета tabulate — обычный текст."""
    try:
        return df.to_markdown(**kw)
    except ImportError:
        return "```\n" + df.to_string(index=kw.get("index", True)) + "\n```"


def fmt_table(df, cols):
    g = df.groupby(["regime", "variant"], sort=False)
    mean, std = g[cols].mean(), g[cols].std().fillna(0)
    out = mean.copy().astype(object)
    for c in cols:
        out[c] = [("—" if np.isnan(m) else f"{m:.3f} ± {s:.3f}") for m, s in zip(mean[c], std[c])]
    return out


def report(rows, boot, chosen, lat, args):
    os.makedirs(args.out, exist_ok=True)
    res, bt, ch, lt = pd.DataFrame(rows), pd.DataFrame(boot), pd.DataFrame(chosen), pd.DataFrame(lat)
    res.to_csv(os.path.join(args.out, "metrics.csv"), index=False)
    bt.to_csv(os.path.join(args.out, "bootstrap.csv"), index=False)
    ch.to_csv(os.path.join(args.out, "chosen.csv"), index=False)
    lt.to_csv(os.path.join(args.out, "latency.csv"), index=False)
    with open(os.path.join(args.out, "args.json"), "w") as f:
        json.dump(vars(args), f, indent=1)

    test = res[res.split == "test"]
    k = args.k
    cols = [c.replace("@10", f"@{k}") for c in SHOW if c.replace("@10", f"@{k}") in test.columns]
    lines = [f"# Результаты: protocol={args.protocol}, window={args.window}, seeds={args.seeds}\n",
             "Тест; среднее ± std по сидам. Квота/порог/параметры TIFU подобраны на val.\n"]
    rs = test.groupby("regime", sort=False)["repeat share (true)"].mean()
    lines.append("Доля повторов среди целей: " + ", ".join(f"{s}: {v:.2f}" for s, v in rs.items()) + "\n")
    lines.append(to_md(fmt_table(test, cols)))
    if len(bt):
        lines.append("\n\n## Парный бутстрап на тесте: ref минус rival\n")
        lines.append("CI — средние по сидам границы 95% интервала; `CI>0` / `CI<0` — в скольких сидах интервал "
                     "целиком выше / ниже нуля. Для `wasted` лучше отрицательная разница.\n")
        for ref, part in bt.groupby("ref", sort=False):
            b = part.groupby(["regime", "rival", "metric"], sort=False).agg(
                diff=("diff", "mean"), ci_lo=("ci_lo", "mean"), ci_hi=("ci_hi", "mean"),
                **{"CI>0": ("ci_lo", lambda x: f"{(x > 0).sum()}/{len(x)}"),
                   "CI<0": ("ci_hi", lambda x: f"{(x < 0).sum()}/{len(x)}")})
            lines.append(f"\n### {ref}\n")
            lines.append(to_md(b, floatfmt=".4f"))
    if len(lt):
        lc = [c for c in ("params", "batch ms/user", "online p50 ms", "online p95 ms", "online p99 ms") if c in lt]
        L = lt.groupby("variant", sort=False)[lc].mean()
        lines.append(f"\n\n## Латентность и стоимость (устройство: {args.device}; среднее по прогонам)\n")
        lines.append("`batch ms/user` — полный расчёт выдачи для всех тестовых пользователей, делённый на их число "
                     f"(батч {args.batch} для нейросетей). `online` — один запрос = один пользователь, "
                     "от истории до топ-k, перцентили по 200 запросам.\n")
        lines.append(to_md(L, floatfmt=".3f"))
    lines.append("\n\n## Подобранные на val параметры\n")
    lines.append(to_md(ch, index=False))
    text = "\n".join(lines)
    with open(os.path.join(args.out, "summary.md"), "w", encoding="utf-8") as f:
        f.write(text)
    print("\n" + text)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--regimes", nargs="+", default=["high", "mid", "low"], choices=list(REGIMES),
                    help="режимы синтетики по доле повторов (см. REGIMES)")
    ap.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
    ap.add_argument("--n-users", type=int, default=4000)
    ap.add_argument("--protocol", default="window", choices=["window", "next", "next-start"])
    ap.add_argument("--t-val", type=float, default=351.0)
    ap.add_argument("--t-test", type=float, default=358.0)
    ap.add_argument("--window", type=float, default=7.0)
    ap.add_argument("--gap", type=float, default=0.0)
    ap.add_argument("--k", type=int, default=10)
    ap.add_argument("--max-len", type=int, default=50)
    ap.add_argument("--dim", type=int, default=64)
    ap.add_argument("--layers", type=int, default=2)
    ap.add_argument("--max-epochs", type=int, default=60)
    ap.add_argument("--patience", type=int, default=3)
    ap.add_argument("--batch", type=int, default=128)
    ap.add_argument("--lr", type=float, default=None, help="общий lr; иначе --lr-base / --lr-switch")
    ap.add_argument("--lr-base", type=float, default=1e-3)
    ap.add_argument("--lr-switch", type=float, default=1e-3)
    ap.add_argument("--switch-variants", nargs="+", default=list(SWITCH_VARIANTS), choices=list(SWITCH_VARIANTS),
                    help="какие варианты repeat-головы обучать (абляция); 'switch' — исходный, обязателен")
    ap.add_argument("--skip-nn", action="store_true")
    ap.add_argument("--out", default="results/main")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = ap.parse_args()
    if "switch" not in args.switch_variants:
        args.switch_variants = ["switch"] + args.switch_variants

    rows, boot, chosen, lat = [], [], [], []
    for regime in args.regimes:
        for seed in args.seeds:
            r, b, c, t = run_one(regime, seed, args)
            rows += r
            boot += b
            chosen.append(c)
            lat += t
    report(rows, boot, chosen, lat, args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile lr_probe.py
"""
Подбор learning rate на val (тест не используется): для каждой модели и lr — лучший val Recall@10 и эпоха.
  python -u lr_probe.py --regime high --seed 0 --lrs 1e-3 3e-3
"""
import argparse
import random

import numpy as np
import torch

from data import make_protocol, make_synthetic
from train import REGIMES, build_train_sequences, train_model

ap = argparse.ArgumentParser()
ap.add_argument("--regime", default="high", choices=list(REGIMES))
ap.add_argument("--seed", type=int, default=0)
ap.add_argument("--lrs", type=float, nargs="+", default=[1e-3, 3e-3])
ap.add_argument("--modes", nargs="+", default=["base", "switch"])
ap.add_argument("--max-epochs", type=int, default=40)
cli = ap.parse_args()

args = argparse.Namespace(k=10, max_len=50, dim=64, layers=2, patience=3, batch=128,
                          max_epochs=cli.max_epochs, device="cpu", lr=None)
cons, dur = REGIMES[cli.regime]
events, items = make_synthetic(consumable_share=cons, durable_rate=dur, seed=cli.seed)
train, val, _ = make_protocol(events)
seqs = build_train_sequences(train, args.max_len)
res = []
for mode in cli.modes:
    for lr in cli.lrs:
        random.seed(cli.seed); np.random.seed(cli.seed); torch.manual_seed(cli.seed)
        args.lr = lr
        _, ep, best = train_model(list(seqs), len(items), mode, args, val, val.users, items)
        res.append((mode, lr, ep, best))
        print(f"### {mode} lr={lr}: best val Recall@10 {best:.4f} at epoch {ep}")
print("\nИтог:")
for mode, lr, ep, best in res:
    print(f"  {mode:7s} lr={lr:g}: val Recall@10 {best:.4f}, эпоха {ep}")


In [ ]:
%%writefile latency_bench.py
"""
Бенчмарк латентности: не зависит от весов, поэтому модели не обучаются — меряется только архитектура
и полный путь выдачи (паддинг → прямой проход → смешивание голов → top-k) на реальных историях.

  python latency_bench.py --device cpu
  python latency_bench.py --device cuda --out results/latency_cuda.csv

Сценарии:
  online — batch=1 (один запрос = один пользователь), перцентили p50/p95/p99 по --n запросам;
  batch  — все тестовые пользователи батчами по --batch, мс на пользователя.
"""
import argparse
import os

import numpy as np
import pandas as pd
import torch

import baselines as bl
from data import make_protocol, make_synthetic
from heuristics import heur_components
from mixing import switch
from model import OneStageRec
from train import REGIMES, SWITCH_VARIANTS, model_kwargs, nn_outputs, online_latency, repeat_prior, timed, variant_prefix
from metrics import topk

ap = argparse.ArgumentParser()
ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
ap.add_argument("--regime", default="high", choices=list(REGIMES))
ap.add_argument("--n", type=int, default=300)
ap.add_argument("--batch", type=int, default=128)
ap.add_argument("--threads", type=int, default=None, help="torch.set_num_threads для CPU")
ap.add_argument("--variants", nargs="+", default=list(SWITCH_VARIANTS), choices=list(SWITCH_VARIANTS))
ap.add_argument("--out", default=None)
cli = ap.parse_args()
if cli.threads:
    torch.set_num_threads(cli.threads)

args = argparse.Namespace(k=10, max_len=50, dim=64, layers=2, batch=cli.batch, device=cli.device)
cons, dur = REGIMES[cli.regime]
events, items = make_synthetic(consumable_share=cons, durable_rate=dur, seed=0)
train, val, test = make_protocol(events)
tu, n_items, k = test.users, len(items), 10
prior = repeat_prior(train, n_items)
print(f"device={cli.device} threads={torch.get_num_threads()} test users={len(tu)}")

rows = []
for var, fn in (("freq+pop", lambda: bl.freq_pop(test, tu, n_items, k)),
                ("tifu-knn", lambda: bl.tifu_knn(test, tu, n_items, k)),
                ("heur/switch", lambda: switch(heur_components(test, tu, items)[1], k))):
    _, sec = timed(fn, args)
    rows.append({"variant": var, "batch ms/user": 1000 * sec / len(tu)})

models = {"nn/base": OneStageRec(n_items, mode="base").to(cli.device)}
for sv in cli.variants:
    kw = model_kwargs(sv, prior)
    full = kw.pop("full_mask", False)
    m = OneStageRec(n_items, mode="switch", **kw).to(cli.device)
    m.full_mask = full
    models[f"{variant_prefix(sv)}/switch"] = m

for name, m in models.items():
    m.eval()
    nn_outputs(m, test, tu[:args.batch], args)                       # прогрев
    out, sec = timed(lambda: nn_outputs(m, test, tu, args), args)
    sec += timed(lambda: topk(out, k) if m.mode == "base" else switch(out, k), args)[1]
    rows.append({"variant": name, "params": sum(p.numel() for p in m.parameters()),
                 "batch ms/user": 1000 * sec / len(tu), **online_latency(m, test, tu, args, n=cli.n)})
    print(rows[-1])

df = pd.DataFrame(rows).set_index("variant")
df.insert(0, "device", cli.device)
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
    print(df)
if cli.out:
    os.makedirs(os.path.dirname(cli.out) or ".", exist_ok=True)
    df.to_csv(cli.out)


## Прогон

In [ ]:
# ---- настройки прогона ----
QUICK = False                 # True — дымовой прогон за пару минут, False — полный
SEEDS = "0 1 2"
REGIMES = "high mid low"        # доля повторов ≈ 0.82–0.88 / 0.23–0.54 / 0.13–0.37 (зависит от сида)
COMMON = "--lr-base 1e-3 --lr-switch 3e-3"   # lr подобраны на val (lr_probe.py)
# абляция: switch — исходный; mask — explore исключает всю историю; bias/null/prior — опции repeat-головы
SWITCH_VARIANTS = "switch mask mask+bias mask+null mask+prior mask+all"
RUN_LEAK_CHECK = False         # цена утечки уже измерена (≈ 0); True — пересчитать
# batch оставлен 128, как на CPU: больший батч меняет оптимизацию и lr пришлось бы подбирать заново

if QUICK:
    SEEDS, COMMON = "0", COMMON + " --n-users 500 --max-epochs 2"
os.makedirs("results", exist_ok=True)
print(f"QUICK={QUICK} SEEDS={SEEDS} REGIMES={REGIMES} COMMON={COMMON}")
print(f"SWITCH_VARIANTS={SWITCH_VARIANTS}")


In [ ]:
# Основной эксперимент: протокол window, все режимы × сиды
t0 = time.time()
run(f"train.py --regimes {REGIMES} --seeds {SEEDS} {COMMON} --switch-variants {SWITCH_VARIANTS} --out results/main",
    "results/main.log")
print(f"основной прогон: {(time.time() - t0) / 60:.1f} мин")


In [ ]:
# Цена утечки времени запроса: старый протокол (время запроса = время покупки) vs та же цель без утечки
if RUN_LEAK_CHECK:
    t0 = time.time()
    for proto, out in (("next", "leak_next"), ("next-start", "leak_next_start")):
        run(f"train.py --protocol {proto} --regimes high --seeds {SEEDS} {COMMON} --switch-variants switch "
            f"--out results/{out}", f"results/{out}.log")
    print(f"проверка утечки: {(time.time() - t0) / 60:.1f} мин")
else:
    print("пропущено (RUN_LEAK_CHECK = False)")


## Результаты

In [ ]:
from IPython.display import Markdown, display
import pandas as pd

display(Markdown(open("results/main/summary.md", encoding="utf-8").read()))

# сводка по утечке: Recall@10 на тесте, среднее по сидам
rows = []
for proto in ("next", "next_start"):
    p = f"results/leak_{proto}/metrics.csv"
    if os.path.exists(p):
        m = pd.read_csv(p)
        m = m[(m.split == "test") & m.variant.isin(["freq+pop", "heur/switch", "nn/base", "nn/switch"])]
        rows.append(m.groupby("variant")["Recall@10"].mean().rename(proto))
if rows:
    leak = pd.concat(rows, axis=1)
    leak["цена утечки"] = leak["next"] - leak["next_start"]
    display(leak.round(4))


In [ ]:
# Латентность на чистой машине: архитектура и путь выдачи, без обучения (веса на задержку не влияют)
n = 50 if QUICK else 300
run(f"latency_bench.py --device cpu --n {n} --out results/latency_cpu.csv", "results/latency_cpu.log")
if torch.cuda.is_available():
    run(f"latency_bench.py --device cuda --n {n} --out results/latency_cuda.csv", "results/latency_cuda.log")
for f in ("results/latency_cpu.csv", "results/latency_cuda.csv"):
    if os.path.exists(f):
        display(pd.read_csv(f).round(3))


In [ ]:
import shutil
archive = shutil.make_archive(os.path.join(BASE, "results"), "zip", "results")
print("сохранено:", archive)
